# 3 Ingeniería de Datos con Pandas
## Importación de librerías

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import csv

## Carga del dataset

In [2]:
''' Al haber detectado antes el problema con las comillas,
    la carga del dataset se hace ignorandolas como separadores '''
df = pd.read_csv(
    "../data/ventas_techventas.csv",
    quoting=csv.QUOTE_NONE,
    encoding="latin1"
)

df.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,"""1003",2024-01-10,C003,DataSolutions,Centro,"Monitor 27""""",Electronica,5,350.0,0.05,"Ana Lï¿½ï¿"""
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


## Información general

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         60 non-null     str    
 1   fecha            60 non-null     str    
 2   cliente_id       60 non-null     str    
 3   cliente_nombre   60 non-null     str    
 4   region           60 non-null     str    
 5   producto         59 non-null     str    
 6   categoria        60 non-null     str    
 7   cantidad         60 non-null     int64  
 8   precio_unitario  60 non-null     float64
 9   descuento        60 non-null     float64
 10  vendedor         60 non-null     str    
dtypes: float64(2), int64(1), str(8)
memory usage: 5.3 KB


## Observaciones de la estructura del dataset

El dataset contiene **60 registros** y **11 columnas** relacionadas con pedidos, clientes, productos y ventas de TechVentas S.A.S.

Se identifican tres tipos principales de datos:

- **Variables categóricas (object):** `order_id`, `cliente_id`, `cliente_nombre`, `region`, `producto`, `categoria` y `vendedor`.
- **Variables numéricas enteras (int64):** `cantidad`.
- **Variables numéricas decimales (float64):** `precio_unitario` y `descuento`.

Durante la exploración inicial se encontró un posible problema de calidad de datos en la columna **producto**, ya que presenta **59 valores no nulos de 60 registros**, lo que indica la existencia de un valor faltante que deberá ser tratado durante la fase de limpieza.

Las columnas numéricas (`cantidad`, `precio_unitario` y `descuento`) no presentan valores nulos según la información mostrada por `info()`. Sin embargo, es necesario realizar validaciones adicionales para detectar posibles valores inconsistentes, como cantidades negativas o descuentos fuera de rango.

La columna `fecha` aparece como tipo `object`, es necesario usar el parámetro `parse_dates` para convertirla a un formato de fecha adecuado y facilitar posteriores análisis temporales y agrupaciones mensuales.

En general, el dataset presenta una estructura consistente y una baja cantidad de datos faltantes, lo que facilita las tareas de limpieza y transformación posteriores. Esto se debe a que en la carga de datos se ignora las comillas dobles que causaba el agrupamiento de información de varias columnas y dejando muchos valores nulos. 

## Detectar nulos con isna()

In [4]:
problemas = df[df.isna().any(axis=1)]

problemas

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,NaN,Electronica,2,350.0,0.05,Maria Torres


## Eliminar/imputar

In [5]:
''' Dentro de los nombres de productos solo existe uno con precio unitario = 350.0 y 
    que sea de la categoria Electronica, por lo tanto, se ubica el registro cuyo nombre 
    del producto es nulo y cumple con las características mencionadas y se imputa el 
    nombre del producto, en este caso "Monitor 27"  '''

df.loc[
    (df["producto"].isna()) &
    (df["precio_unitario"] == 350.0) &
    (df["categoria"] == "Electronica"),
    "producto"
] = "Monitor 27"

## luego del proceso Eliminar/imputar se descartaron 0 registros, puesto que el único registro con problemas fue recuperado usando la información disponible

##  Crear columnas ingreso_bruto, ingreso_neto y mes.

In [6]:
df["ingreso_bruto"] = (
    df["cantidad"] *
    df["precio_unitario"]
)
df["ingreso_neto"] = (
    df["ingreso_bruto"] *
    (1 - df["descuento"])
)
# Normalizar fechas para poder crear mes

df["fecha"] = pd.to_datetime(
    df["fecha"],
    errors="coerce"
)

df = df[df["fecha"].notna()]
df["mes"] = df["fecha"].dt.to_period("M")

df[
    [
        "fecha",
        "cantidad",
        "precio_unitario",
        "ingreso_bruto",
        "ingreso_neto",
        "mes"
    ]
].head()

,fecha,cantidad,precio_unitario,ingreso_bruto,ingreso_neto,mes
0,2024-01-05,2,1200.0,2400.0,2160.0,2024-01
1,2024-01-07,15,25.0,375.0,375.0,2024-01
2,2024-01-10,5,350.0,1750.0,1662.5,2024-01
3,2024-01-12,10,85.0,850.0,765.0,2024-01
4,2024-01-15,3,1200.0,3600.0,3060.0,2024-01


## Agrupación mensual

In [7]:
resumen = (
    df.groupby("mes")
      .agg(
          total=("ingreso_neto", "sum"),
          n=("order_id", "count")
      )
      .reset_index()
)

resumen

,mes,total,n
0,2024-01,13773.50,10
1,2024-02,13050.00,10
2,2024-03,14454.00,11
3,2024-04,16689.00,11
4,2024-05,11387.25,11
5,2024-06,11209.25,7


## Merge con tabla de metas

In [8]:
# Crear tabla de metas
metas = pd.DataFrame({
    "mes": resumen["mes"],
    "meta": [
        10000,
        10000,
        10000,
        10000,
        10000,
        10000
    ][:len(resumen)]
})

metas

,mes,meta
0,2024-01,10000
1,2024-02,10000
2,2024-03,10000
3,2024-04,10000
4,2024-05,10000
5,2024-06,10000


## Merge

In [9]:
cumplimiento = pd.merge(
    resumen,
    metas,
    on="mes",
    how="left"
)

cumplimiento

,mes,total,n,meta
0,2024-01,13773.50,10,10000
1,2024-02,13050.00,10,10000
2,2024-03,14454.00,11,10000
3,2024-04,16689.00,11,10000
4,2024-05,11387.25,11,10000
5,2024-06,11209.25,7,10000


## Calcular porcentaje

In [10]:
cumplimiento["porcentaje_cumplimiento"] = (
    cumplimiento["total"] /
    cumplimiento["meta"]
) * 100

cumplimiento

,mes,total,n,meta,porcentaje_cumplimiento
0,2024-01,13773.50,10,10000,137.7350
1,2024-02,13050.00,10,10000,130.5000
2,2024-03,14454.00,11,10000,144.5400
3,2024-04,16689.00,11,10000,166.8900
4,2024-05,11387.25,11,10000,113.8725
5,2024-06,11209.25,7,10000,112.0925


# Exportación SQLite
## Exportar

In [16]:
engine = create_engine(
    "sqlite:///../output/techventas.db"
)
# Se convierte el campo mes a string para evitar problemas al exportar a SQL
df["mes"] = df["mes"].astype(str)
df.to_sql(
    "ventas",
    engine,
    if_exists="replace",
    index=False
)

print("Datos exportados correctamente")

Datos exportados correctamente


##  verifica leyendo de vuelta con read_sql

In [17]:
verificacion = pd.read_sql(
    "SELECT COUNT(*) FROM ventas",
    engine
)

verificacion

,COUNT(*)
0,60


la verificación fue correcta, puesto que no se descartaron registros